# Fine-tune a small open model on the channel transcripts

Runs on Colab's **free T4 GPU** using [Unsloth](https://github.com/unslothai/unsloth) for QLoRA fine-tuning.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

Steps: install deps -> upload the two JSONL files produced locally by `prepare_finetune_data.py` -> load a 4-bit base model -> attach LoRA adapters -> train -> smoke-test -> export to GGUF -> download the GGUF for use with Ollama.

## 1. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "git+https://github.com/unslothai/unsloth.git"

## 2. Upload the training data
Upload `finetune_train.jsonl` and `finetune_val.jsonl` (created locally by `prepare_finetune_data.py`).

In [ ]:
from google.colab import files

print("Select finetune_train.jsonl and finetune_val.jsonl")
uploaded = files.upload()

# Alternative: mount Google Drive instead of uploading each session, e.g.:
# from google.colab import drive
# drive.mount('/content/drive')
# TRAIN_PATH = "/content/drive/MyDrive/finetune_train.jsonl"
# VAL_PATH = "/content/drive/MyDrive/finetune_val.jsonl"

TRAIN_PATH = "finetune_train.jsonl"
VAL_PATH = "finetune_val.jsonl"

## 3. Load the base model (4-bit)
`unsloth/Meta-Llama-3.1-8B-bnb-4bit` fits comfortably on a free T4. Swap in `unsloth/mistral-7b-bnb-4bit` here if you'd rather use Mistral.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,       # auto-detect (bfloat16 on T4-supported hardware, else float16)
    load_in_4bit=True,
)

## 4. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 5. Load and format the dataset
Renders each example's `messages` through the model's chat template into a single training string.

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset(
    "json",
    data_files={"train": TRAIN_PATH, "validation": VAL_PATH},
)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

formatted_datasets = raw_datasets.map(format_example, remove_columns=raw_datasets["train"].column_names)
print(formatted_datasets)
print(formatted_datasets["train"][0]["text"][:500])

## 6. Train
Batch size + gradient accumulation are sized to fit a free-tier T4 (16GB).

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_datasets["train"],
    eval_dataset=formatted_datasets["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="no",
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=3407,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 7. Smoke-test the fine-tuned model
Ask it something in-topic and eyeball whether the voice/style landed.

In [ ]:
FastLanguageModel.for_inference(model)

PERSONA_SYSTEM_PROMPT = (
    "You are a direct, no-nonsense relationship and dating coach who speaks "
    "candidly to men about modern relationships, dating, and marriage. You "
    "back up your points with references to evolutionary psychology and "
    "biology, break advice down into clear numbered points, speak with blunt "
    "confidence, address the listener as \"guys\", and close out with a "
    "motivational, no-excuses tone."
)

test_messages = [
    {"role": "system", "content": PERSONA_SYSTEM_PROMPT},
    {"role": "user", "content": "Give me your take on: dating someone with a lot of emotional baggage"},
]

inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

output = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.8, do_sample=True)
print(tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True))

## 8. Export to GGUF (for Ollama)
`q4_k_m` keeps the file small enough to download while still running well in Ollama.

In [ ]:
model.save_pretrained_gguf("laurin-model", tokenizer, quantization_method="q4_k_m")

## 9. Download the GGUF file

In [ ]:
import glob

gguf_files = glob.glob("laurin-model*.gguf") + glob.glob("laurin-model/*.gguf")
print("Found:", gguf_files)

for path in gguf_files:
    files.download(path)